#### Código para comparação entre modelos utilizando throughput_rate e departure_rate

- Código 1: Comparação modelo contínuo;
- Código 2: Comparação modelo discreto.

In [ ]:
pip install numpy gplearn sympy pysindy pandas scikit-learn

In [ ]:
import numpy as np
import pandas as pd
import pysindy as ps
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# 1. Carregamento e Preparação Inicial
url = 'https://raw.githubusercontent.com/Surtivo/SR_PySINDy/refs/heads/main/ns3_simulation_queue_metrics.csv'
df_raw = pd.read_csv(url)

L = 1500  # MTU em bytes
horizontes = [1, 3, 5, 10]

# Função de Rollout Reutilizável
def evaluate_rollout(model, X_block, df_block, steps=1, dt=0.1, use_queue_full=False, enforce_queue_cap=False, q_max=20.0):
    U_base = df_block[["net_arrival_pps", "throughput_pps"]].values
    num_samples = len(X_block) - steps
    q_preds, q_trues = [], []

    for i in range(num_samples):
        q_sim = X_block[i, 0]
        for s in range(steps):
            u_in_out = U_base[i + s]
            if use_queue_full:
                q_full_val = 1.0 if q_sim >= q_max else 0.0
                u_step = np.append(u_in_out, q_full_val)
            else:
                u_step = u_in_out

            dq_dt = model.predict(np.array([[q_sim]]), u=np.array([u_step]))[0, 0]
            q_sim = q_sim + dq_dt * dt

            if enforce_queue_cap:
                q_sim = np.clip(q_sim, 0.0, q_max)
            else:
                q_sim = max(0.0, q_sim)

        q_preds.append(q_sim)
        q_trues.append(X_block[i + steps, 0])

    r2 = r2_score(q_trues, q_preds)
    rmse = np.sqrt(mean_squared_error(q_trues, q_preds))
    mae = mean_absolute_error(q_trues, q_preds)
    return r2, rmse, mae

# Função para Executar o Grid de Modelos
def run_experiments(df_input, target_saida_col):
    df = df_input.copy()
    df["net_arrival_pps"] = (df["arrival_rate_mbps"] * 10**6) / (8 * L)
    df["throughput_pps"] = (df[target_saida_col] * 10**6) / (8 * L)
    df["queue_full"] = (df["queue_packets"] >= 20).astype(float)

    n_train = int(len(df) * 0.8)
    df_train = df.iloc[:n_train].copy()
    df_val = df.iloc[n_train:].copy()

    t_train = df_train["time"].values
    X_train = df_train[["queue_packets"]].values
    X_val = df_val[["queue_packets"]].values

    results = []

    for grau in range(1, 4):
        for use_qfull in [False, True]:
            for use_cap in [False, True]:
                if use_qfull:
                    cols_u = ["net_arrival_pps", "throughput_pps", "queue_full"]
                    f_names = ["q", "net_arrival_pps", "throughput_pps", "queue_full"]
                else:
                    cols_u = ["net_arrival_pps", "throughput_pps"]
                    f_names = ["q", "net_arrival_pps", "throughput_pps"]

                U_train = df_train[cols_u].values

                feature_lib = ps.PolynomialLibrary(degree=grau, include_bias=False)
                model = ps.SINDy(
                    differentiation_method=ps.SmoothedFiniteDifference(),
                    feature_library=feature_lib,
                    optimizer=ps.STLSQ(threshold=0.0001)
                ).fit(X_train, t=t_train, u=U_train, feature_names=f_names)

                coefs = model.coefficients()[0]
                n_terms = np.sum(coefs != 0)

                row = {
                    "Grau": grau,
                    "Q_Full": "Sim" if use_qfull else "Não",
                    "Cap": "Sim" if use_cap else "Não",
                    "Termos": n_terms
                }

                for h in horizontes:
                    r2, rmse, mae = evaluate_rollout(
                        model, X_val, df_val, steps=h,
                        use_queue_full=use_qfull, enforce_queue_cap=use_cap
                    )
                    row[f"R² ({h}s)"] = r2
                    row[f"RMSE ({h}s)"] = rmse
                    row[f"MAE ({h}s)"] = mae

                results.append(row)
        # print("========================== Equação Grau:", grau, "- Alvo:", target_saida_col, "============================",)
        # model.print()
        # print("==========================================================================================\n")

    return pd.DataFrame(results)

# 2. Execução das Duas Tabelas Comparativas
df_res_departure = run_experiments(df_raw, "departure_rate_mbps")
df_res_throughput = run_experiments(df_raw, "throughput_mbps")

# 3. Exibição dos Resultados
print("==========================================================================================")
print("=== TABELA 1: USANDO departure_rate_mbps (Taxa de Saída Real Bruta) ===")
print("==========================================================================================")
print(df_res_departure.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

print("\n==========================================================================================")
print("=== TABELA 2: USANDO throughput_mbps (Vazão Efetiva Utilizada) ===")
print("==========================================================================================")
print(df_res_throughput.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

In [ ]:
import numpy as np
import pandas as pd
import pysindy as ps
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# 1. Carregamento e Preparação Inicial
url = 'https://raw.githubusercontent.com/Surtivo/SR_PySINDy/refs/heads/main/ns3_simulation_queue_metrics.csv'
df_raw = pd.read_csv(url)

L = 1500  # MTU em bytes
horizontes = [1, 3, 5, 10]

# Função de Rollout Autoregressivo Discreto
def evaluate_discrete_rollout(model, X_block, df_block, steps=1, use_queue_full=False, enforce_queue_cap=False, q_max=20.0):
    U_base = df_block[["net_arrival_pps", "throughput_pps"]].values
    num_samples = len(X_block) - steps
    q_preds, q_trues = [], []

    for i in range(num_samples):
        q_sim = X_block[i, 0]

        # Iteração por passos discretos: q[k+1] = f(q[k], u[k])
        for s in range(steps):
            u_in_out = U_base[i + s]
            if use_queue_full:
                q_full_val = 1.0 if q_sim >= q_max else 0.0
                u_step = np.append(u_in_out, q_full_val)
            else:
                u_step = u_in_out

            # Previsão direta do próximo estado q[k+1]
            q_next = model.predict(np.array([[q_sim]]), u=np.array([u_step]))[0, 0]

            # Restrições físicas
            if enforce_queue_cap:
                q_sim = np.clip(q_next, 0.0, q_max)
            else:
                q_sim = max(0.0, q_next)

        q_preds.append(q_sim)
        q_trues.append(X_block[i + steps, 0])

    r2 = r2_score(q_trues, q_preds)
    rmse = np.sqrt(mean_squared_error(q_trues, q_preds))
    mae = mean_absolute_error(q_trues, q_preds)
    return r2, rmse, mae

# Função para Executar a Varredura Discreta
def run_discrete_experiments(df_input, target_saida_col):
    df = df_input.copy()
    df["net_arrival_pps"] = (df["arrival_rate_mbps"] * 10**6) / (8 * L)
    df["throughput_pps"] = (df[target_saida_col] * 10**6) / (8 * L)
    df["queue_full"] = (df["queue_packets"] >= 20).astype(float)

    n_train = int(len(df) * 0.8)
    df_train = df.iloc[:n_train].copy()
    df_val = df.iloc[n_train:].copy()

    # Prepara os pares de transição de tempo discreto: x[k] -> x[k+1]
    X_train_k = df_train[["queue_packets"]].values[:-1]
    X_train_k1 = df_train[["queue_packets"]].values[1:]
    t_train_k = df_train["time"].values[:-1]

    X_val = df_val[["queue_packets"]].values

    results = []

    for grau in range(1, 4):
        for use_qfull in [False, True]:
            for use_cap in [False, True]:
                if use_qfull:
                    cols_u = ["net_arrival_pps", "throughput_pps", "queue_full"]
                    f_names = ["q", "net_arrival_pps", "throughput_pps", "queue_full"]
                else:
                    cols_u = ["net_arrival_pps", "throughput_pps"]
                    f_names = ["q", "net_arrival_pps", "throughput_pps"]

                U_train_k = df_train[cols_u].values[:-1]

                feature_lib = ps.PolynomialLibrary(degree=grau, include_bias=False)

                model = ps.SINDy(
                    differentiation_method=None,
                    feature_library=feature_lib,
                    optimizer=ps.STLSQ(threshold=0.0001)
                )

                # Passa o 't=t_train_k' obrigatório e sobrescreve o alvo com 'x_dot=X_train_k1'
                model.fit(X_train_k, t=t_train_k, x_dot=X_train_k1, u=U_train_k, feature_names=f_names)

                coefs = model.coefficients()[0]
                n_terms = np.sum(coefs != 0)

                row = {
                    "Grau": grau,
                    "Q_Full": "Sim" if use_qfull else "Não",
                    "Cap": "Sim" if use_cap else "Não",
                    "Termos": n_terms
                }

                for h in horizontes:
                    r2, rmse, mae = evaluate_discrete_rollout(
                        model, X_val, df_val, steps=h,
                        use_queue_full=use_qfull, enforce_queue_cap=use_cap
                    )
                    row[f"R² ({h}s)"] = r2
                    row[f"RMSE ({h}s)"] = rmse
                    row[f"MAE ({h}s)"] = mae

                results.append(row)

    return pd.DataFrame(results)

# 2. Execução das Tabelas Comparativas Discretas
df_res_discrete_departure = run_discrete_experiments(df_raw, "departure_rate_mbps")
df_res_discrete_throughput = run_discrete_experiments(df_raw, "throughput_mbps")

# 3. Exibição dos Resultados
print("==========================================================================================")
print("=== TABELA 1: DISCRETESINDY COM departure_rate_mbps (Taxa de Saída Real Bruta) ===")
print("==========================================================================================")
print(df_res_discrete_departure.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

print("\n==========================================================================================")
print("=== TABELA 2: DISCRETESINDY COM throughput_mbps (Vazão Efetiva Utilizada) ===")
print("==========================================================================================")
print(df_res_discrete_throughput.to_string(index=False, float_format=lambda x: f"{x:.4f}"))